In [14]:
import sqlite3
import pandas as pd
from helpers import Is_english
from tqdm import tqdm
tqdm.pandas()

bias_df = pd.read_csv("allsides-ranking.csv")

db_path = "data/news_articles.db"  
conn = sqlite3.connect(db_path)

In [2]:
# Add bias column to table - commented out because after adding it cant be added again

#conn.execute("ALTER TABLE article_urls ADD COLUMN bias TEXT;")

In [3]:
# Populate Bias column with data from bias df

cur = conn.cursor()

for _, row in bias_df.iterrows():
    cur.execute("""
        UPDATE article_urls
        SET bias = ?
        WHERE lower(outlet_name) = lower(?)
    """, (row["bias"], row["outlet"]))

conn.commit()

In [4]:
# article_urls table info
print("article_urls")
display(pd.read_sql("PRAGMA table_info(article_urls);", conn))

# article_contents table info
print("article_contents")
display(pd.read_sql("PRAGMA table_info(article_contents);", conn))

article_urls


,cid,name,type,notnull,dflt_value,pk
0,0,uuid,TEXT,0,None,0
1,1,url,TEXT,0,None,0
2,2,outlet_name,TEXT,0,None,0
3,3,bias,TEXT,0,None,0


article_contents


,cid,name,type,notnull,dflt_value,pk
0,0,uuid,TEXT,0,None,0
1,1,date,TEXT,0,None,0
2,2,content,TEXT,0,None,0
3,3,content_preprocessed,TEXT,0,None,0
4,4,language,TEXT,0,None,0


In [5]:
# filter to number of articles per year to find viable outlets

query_long = """
SELECT
  u.outlet_name,
  COALESCE(u.bias, 'Unknown') AS bias,
  SUBSTR(c.date,1,4) AS year,
  COUNT(*) AS n
FROM article_urls u
JOIN article_contents c
  ON u.uuid = c.uuid
WHERE c.date IS NOT NULL
  AND c.date >= '2015-01-01'
  AND c.date <  '2022-01-01'
GROUP BY u.outlet_name, COALESCE(u.bias,'Unknown'), SUBSTR(c.date,1,4);
"""
df_long = pd.read_sql(query_long, conn)

# Pivot to years as columns
df_pivot = (df_long.pivot_table(index=['outlet_name','bias'],columns='year', values='n',fill_value=0).reset_index())
year_cols = [c for c in df_pivot.columns if c.isdigit()]
df_pivot[year_cols] = df_pivot[year_cols].astype(int)

In [6]:
df_pivot = df_pivot.sort_values('2021', ascending=False)
df_pivot.head(40)

year,outlet_name,bias,2015,2016,2017,2018,2019,2020,2021
19,Newsweek,Left,2180,2843,3964,4576,4018,5210,9547
4,Breitbart News,Right,1684,2396,2091,2495,3133,4493,6095
18,Newsmax,Unknown,1202,789,957,1078,395,1004,5288
17,New York Post,Unknown,633,717,847,2209,1777,3118,3855
7,CNBC,Center,880,801,1348,1532,2040,3229,3646
9,Daily Beast,Left,1936,2023,1623,2219,2680,3887,2706
42,Washington Times,Lean Right,1145,759,1672,1547,1345,1870,2525
29,The Epoch Times,Lean Right,128,177,27,82,638,166,2413
34,The Washington Post,Unknown,1295,1375,1426,1019,824,900,2310
40,Vox,Left,7824,4620,4332,3368,2083,3164,2010


In [15]:
# Pulling additional outlets selected based on political bias and next largest sample sizes

OUTLETS = (
    "Vox",
    "Red State",
    "Vice",
    "Wall Street Journal",
    "Slate",
    "The New Yorker",
    "Townhall",
    "Bizpac",
    "The Gateway Pundit",
    "PJ Media",
    "MSNBC",
    "Orange County Register",
    "The Guardian",
    "Fox News"
)

query = f"""
SELECT 
    u.uuid,
    u.url,
    u.outlet_name,
    u.bias,
    c.date,
    c.content,
    c.content_preprocessed
FROM article_urls u
JOIN article_contents c
    ON u.uuid = c.uuid
WHERE c.date IS NOT NULL
  AND c.date >= '2015-01-01'
  AND c.date <  '2022-01-01'
  AND u.outlet_name IN ({','.join(['?']*len(OUTLETS))});
"""

LRC_articles = pd.read_sql(query, conn, params=list(OUTLETS))
LRC_articles.head()

,uuid,url,outlet_name,bias,date,content,content_preprocessed
0,16541231938716086625,https://www.vox.com/the-goods/2019/2/7/1821562...,Vox,Left,2019-02-07,Walgreens is participating in a pilot of Coole...,Walgreens is participating in a pilot of Coole...
1,4779102715220679664,https://www.vox.com/2015/2/26/11559408/exclusi...,Vox,Left,2015-02-26,"When my husband, Kevin, and I moved in togethe...","When my husband, Kevin, and I moved in togethe..."
2,1112422139534214672,https://www.vox.com/2017/8/1/16074808/facebook...,Vox,Left,2017-08-01,"The likes of Amazon, Facebook and Google are a...","The likes of Amazon, Facebook and Google are a..."
3,12895774758268355422,https://www.vox.com/2020/6/23/21300563/coronav...,Vox,Left,2020-06-23,"Dr. Anthony Fauci, the nation’s top infectious...","Dr. Anthony Fauci, the nation’s top infectious..."
4,8833566516430940259,https://www.vox.com/2021/1/16/22234971/trump-t...,Vox,Left,2021-01-16,In the wake of the deadly January 6 riot at th...,In the wake of the deadly January 6 riot at th...


In [16]:
LRC_articles.isna().sum()

uuid                        0
url                         0
outlet_name                 0
bias                    16008
date                        0
content                  5339
content_preprocessed        0
dtype: int64

In [17]:
LRC_articles[LRC_articles["bias"].isna()]["outlet_name"].unique()

array(['Wall Street Journal', 'Fox News'], dtype=object)

In [18]:
# Update political bias where missing due to naming convention 

bias_fix = {
    "Wall Street Journal": "Center",
    "Fox News": "Right",
}

LRC_articles["bias"] = LRC_articles.apply(
    lambda row: bias_fix.get(row["outlet_name"], row["bias"]),
    axis=1
)

LRC_articles.isna().sum()

uuid                       0
url                        0
outlet_name                0
bias                       0
date                       0
content                 5339
content_preprocessed       0
dtype: int64

In [19]:
# Drop records with no content
LRC_articles = LRC_articles.dropna()

In [20]:
LRC_articles.isna().sum()

uuid                    0
url                     0
outlet_name             0
bias                    0
date                    0
content                 0
content_preprocessed    0
dtype: int64

In [21]:
# Drop records under 100 words or in a foreign language
LRC_articles = LRC_articles[LRC_articles["content"].str.split().str.len() >=100]
LRC_articles = LRC_articles[LRC_articles["content"].progress_apply(Is_english)]    

100%|███████████████████████████████████████████████████████████████████████████| 77229/77229 [12:01<00:00, 107.05it/s]


In [22]:
LRC_articles.groupby("outlet_name").size()

outlet_name
Fox News                   2050
MSNBC                      4161
Orange County Register     1814
PJ Media                   5210
Red State                  4901
Slate                      4880
The Gateway Pundit         2072
The Guardian               1930
The New Yorker             6188
Townhall                   1556
Vice                      11251
Vox                       26308
Wall Street Journal        4832
dtype: int64

In [23]:
LRC_articles.to_parquet("data/LRC_articles_p2.parquet", index=False)